In [10]:
#==================================
# LandCover 分类占比提取脚本
# Author : King
# Date : 2026-02-21
# 说明 : 提取landcover栅格在5km缓冲区内的分类占比，并输出为CSV文件
#==================================




# ==================== 1. 参数配置 ====================
# Landcover 栅格路径
lc_raster = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\MyProject8.gdb\Conbine_5km"

# 5km 缓冲区图层路径
buffer_layer = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\MyProject8.gdb\Buffer_5km"

# JSON 映射字典路径 (数字代码对应的实际名称)
mapping_json = r"C:\Users\King\Desktop\大三下学习生活\prj-Alban\data\raw\landcover\lc_mapping.json"

# CSV 结果输出路径
output_csv = r"C:\Users\King\Desktop\大三下学习生活\prj-Alban\data\output\LandCover_Stats_Final.csv"


In [ ]:


import arcpy
import os
import pandas as pd
import json
import time
import sys


# ==================== 2. 环境初始化 ====================
arcpy.env.overwriteOutput = True
arcpy.env.extent = None  # 确保全局范围清空

if arcpy.CheckExtension("Spatial") == "Available":
    arcpy.CheckOutExtension("Spatial")
else:
    print("错误：无法获取 Spatial Analyst 许可，脚本终止。")
    sys.exit()

try:
    with open(mapping_json, 'r', encoding='utf-8') as f:
        lc_map = json.load(f)
except Exception as e:
    print(f"读取 JSON 失败: {e}")
    sys.exit()

# ==================== 3. 任务准备 ====================
arcpy.management.SelectLayerByAttribute(buffer_layer, "CLEAR_SELECTION")
total_points = int(arcpy.management.GetCount(buffer_layer)[0])
desc = arcpy.Describe(buffer_layer)
oid_fieldname = desc.OIDFieldName

all_results = []
start_time = time.time()
processed_count = 0
consecutive_errors = 0

print(f"--- 提取任务启动：共计 {total_points} 个观测点 ---")

# 【核心机制】：在内存中创建一个虚拟图层，专门用来“选中”单一个点
tool_lyr = "Buffer_Virtual_Layer"
arcpy.management.MakeFeatureLayer(buffer_layer, tool_lyr)

# ==================== 4. 核心执行逻辑 ====================
with arcpy.da.SearchCursor(buffer_layer, ["SHAPE@", oid_fieldname]) as cursor:
    for row in cursor:
        buffer_geom = row[0]
        oid_val = row[1]
        
        try:
            arcpy.env.extent = buffer_geom.extent
            temp_table = r"memory\lc_tab_temp"
            if arcpy.Exists(temp_table):
                arcpy.management.Delete(temp_table)
            
            # 【最关键一步】：只选中当前 OID 的那唯一一个多边形
            where_clause = f"{oid_fieldname} = {oid_val}"
            arcpy.management.SelectLayerByAttribute(tool_lyr, "NEW_SELECTION", where_clause)
            
            # 面积制表：因为图层里只选中了1个点，输出必然只有1行
            arcpy.sa.TabulateArea(tool_lyr, oid_fieldname, lc_raster, "Value", temp_table)
            
            fields = [f.name for f in arcpy.ListFields(temp_table) if f.name.startswith("VALUE_")]
            
            with arcpy.da.SearchCursor(temp_table, fields) as t_cursor:
                for t_row in t_cursor:
                    total_area = sum(t_row)
                    if total_area > 0:
                        record = {"OID": oid_val}
                        for i, f_name in enumerate(fields):
                            val_code = str(f_name.split("_")[-1])
                            raw_name = lc_map.get(val_code, f"Class_Code_{val_code}")
                            clean_name = "".join(c if c.isalnum() else "_" for c in raw_name)
                            record[clean_name] = round(t_row[i] / total_area, 4)
                            
                        all_results.append(record)

            processed_count += 1
            consecutive_errors = 0 
            
            if processed_count % 5 == 0 or processed_count == total_points:
                elapsed = time.time() - start_time
                speed = processed_count / elapsed 
                percent = (processed_count / total_points) * 100
                remaining = total_points - processed_count
                eta_min = (remaining / speed / 60) if speed > 0 else 0
                print(f"进度: {percent:.1f}% ({processed_count}/{total_points}) | 速度: {speed:.2f} 点/秒 | 预计剩余: {eta_min:.1f} 分钟")

            arcpy.management.Delete(temp_table)

        except Exception as e:
            consecutive_errors += 1
            print(f"!!! OID {oid_val} 处理失败: {e} (连续失败: {consecutive_errors})")
            if consecutive_errors >= 3:
                print("\n[熔断] 连续 3 次提取失败，程序安全终止。")
                break

# ==================== 5. 数据导出 ====================
if all_results:
    print(f"\n正在整合并导出数据...")
    df = pd.DataFrame(all_results).fillna(0)
    cols = ["OID"] + [c for c in df.columns if c != "OID"]
    df = df[cols]
    
    df.to_csv(output_csv, index=False)
    total_time = (time.time() - start_time) / 60
    print(f"--- 任务圆满完成 ---")
    print(f"成功提取: {len(df)} 行标准数据")
    print(f"CSV 保存至: {output_csv}")
else:
    print("错误：未采集到有效数据。")

# 清理
arcpy.CheckInExtension("Spatial")
arcpy.env.extent = None
if arcpy.Exists(tool_lyr):
    arcpy.management.Delete(tool_lyr)

--- 提取任务启动：共计 202 个观测点 ---
进度: 2.5% (5/202) | 速度: 0.81 点/秒 | 预计剩余: 4.1 分钟
进度: 5.0% (10/202) | 速度: 0.71 点/秒 | 预计剩余: 4.5 分钟
进度: 7.4% (15/202) | 速度: 0.64 点/秒 | 预计剩余: 4.9 分钟
进度: 9.9% (20/202) | 速度: 0.55 点/秒 | 预计剩余: 5.5 分钟
进度: 12.4% (25/202) | 速度: 0.52 点/秒 | 预计剩余: 5.7 分钟
进度: 14.9% (30/202) | 速度: 0.53 点/秒 | 预计剩余: 5.4 分钟
进度: 17.3% (35/202) | 速度: 0.55 点/秒 | 预计剩余: 5.1 分钟
进度: 19.8% (40/202) | 速度: 0.56 点/秒 | 预计剩余: 4.8 分钟
进度: 22.3% (45/202) | 速度: 0.56 点/秒 | 预计剩余: 4.7 分钟
进度: 24.8% (50/202) | 速度: 0.56 点/秒 | 预计剩余: 4.5 分钟
进度: 27.2% (55/202) | 速度: 0.56 点/秒 | 预计剩余: 4.4 分钟
进度: 29.7% (60/202) | 速度: 0.56 点/秒 | 预计剩余: 4.2 分钟
进度: 32.2% (65/202) | 速度: 0.57 点/秒 | 预计剩余: 4.0 分钟
进度: 34.7% (70/202) | 速度: 0.58 点/秒 | 预计剩余: 3.8 分钟
进度: 37.1% (75/202) | 速度: 0.59 点/秒 | 预计剩余: 3.6 分钟
进度: 39.6% (80/202) | 速度: 0.60 点/秒 | 预计剩余: 3.4 分钟
进度: 42.1% (85/202) | 速度: 0.61 点/秒 | 预计剩余: 3.2 分钟
